# ALPHA — voix éton provisoires avec OmniVoice

Ce notebook fabrique les **clips audio éton** d'ALPHA avec [OmniVoice](https://github.com/k2-fsa/OmniVoice) (k2-fsa, licence Apache-2.0, langue `eto`).
Les clips sont générés **ici, une seule fois** : l'extension ne fera que lire des fichiers audio, sans modèle et sans connexion.

**Ce sont des voix provisoires.** Le locuteur doit écouter chaque clip (étape 7). Ses propres enregistrements remplaceront ces fichiers.

Étapes : 1) GPU → 2) installation → 3) dossier de travail → 4) textes → 5) modèle → 6) choix de la voix → 7) génération et écoute → 8) téléchargement.

Lancez les cellules **une par une, dans l'ordre** (bouton ▶ à gauche de chaque cellule).

## 1. Vérifier le GPU
Si la cellule affiche « Pas de GPU » : menu **Exécution → Modifier le type d'exécution → GPU T4 → Enregistrer**, puis relancez cette cellule.

In [ ]:
import subprocess
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True, check=True).stdout)
except Exception:
    print("⚠️ Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4")

## 2. Installer OmniVoice
Environ 2 à 4 minutes. Si Colab demande de **redémarrer la session**, acceptez, puis reprenez à l'étape 3.

In [ ]:
!pip install -q omnivoice
import torch
print("PyTorch", torch.__version__, "| GPU disponible :", torch.cuda.is_available())

## 3. Dossier de travail
Avec `SAUVER_SUR_DRIVE = True`, les clips sont enregistrés dans votre Google Drive (dossier **ALPHA_omnivoice**) : si Colab se déconnecte, rien n'est perdu et la génération reprend où elle s'était arrêtée.

In [ ]:
import os
SAUVER_SUR_DRIVE = True

if SAUVER_SUR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    TRAVAIL = "/content/drive/MyDrive/ALPHA_omnivoice"
else:
    TRAVAIL = "/content/ALPHA_omnivoice"

WAV = os.path.join(TRAVAIL, "wav")
OGG = os.path.join(TRAVAIL, "ogg")
os.makedirs(WAV, exist_ok=True)
os.makedirs(OGG, exist_ok=True)
print("Dossier de travail :", TRAVAIL)

## 4. Envoyer les textes éton
Sur votre PC, dans le dossier ALPHA, lancez d'abord `node tools/export-eton-tts.js`.
Puis exécutez cette cellule et choisissez le fichier **docs/audio/eton-tts.jsonl**.

In [ ]:
import json
from google.colab import files

envoi = files.upload()
nom = next(iter(envoi))
TEXTES = [json.loads(l) for l in envoi[nom].decode("utf-8").splitlines() if l.strip()]
with open(os.path.join(TRAVAIL, "eton-tts.jsonl"), "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(t, ensure_ascii=False) for t in TEXTES) + "\n")
print(len(TEXTES), "textes éton chargés. Exemples :")
for t in TEXTES[:5]:
    print(" -", t["id"], ":", t["text"])

# Textes modifiés depuis la dernière génération (« regenerer ») : on efface l'ancien clip pour le refaire.
refaire = [t for t in TEXTES if t.get("regenerer")]
for t in refaire:
    for fichier in (os.path.join(OGG, t["id"] + ".ogg"), os.path.join(WAV, t["id"] + ".wav")):
        if os.path.exists(fichier):
            os.remove(fichier)
print(len(refaire), "clip(s) à refaire car leur texte a changé.")

## 5. Charger le modèle
Environ 1 à 2 minutes au premier lancement (téléchargement du modèle).

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf

model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
try:
    langues = model.supported_language_ids
    langues = langues() if callable(langues) else langues
    print("Éton (eto) pris en charge :", "eto" in langues)
except Exception as erreur:
    print("Liste des langues indisponible :", erreur)
FREQ = 24000  # OmniVoice produit de l'audio à 24 kHz

## 6. Choisir la voix
Sans voix de référence, OmniVoice prend une voix **différente à chaque phrase**. On fixe donc une seule voix pour tous les clips.

**Option A — voix du locuteur** (seulement avec son accord) : mettez `VOIX_LOCUTEUR = True`, envoyez un extrait **de 3 à 10 secondes en éton**, propre, sans bruit, et écrivez dans `TEXTE_EXTRAIT` exactement ce qu'il dit.

**Option B — voix de synthèse** : laissez `VOIX_LOCUTEUR = False`. La cellule propose 4 voix ; écoutez-les et notez le numéro préféré à l'étape 6 bis.

In [ ]:
from IPython.display import Audio, display

VOIX_LOCUTEUR = False
TEXTE_EXTRAIT = ""  # Option A : transcription exacte de l'extrait du locuteur

PHRASE_TEST = max(TEXTES, key=lambda t: len(t["text"]))["text"]  # la phrase la plus longue
REFERENCE = os.path.join(TRAVAIL, "voix_reference.wav")

if VOIX_LOCUTEUR:
    envoi = files.upload()
    chemin = os.path.join(TRAVAIL, "extrait_locuteur" + os.path.splitext(next(iter(envoi)))[1])
    with open(chemin, "wb") as f:
        f.write(next(iter(envoi.values())))
    REFERENCE, TEXTE_REFERENCE = chemin, TEXTE_EXTRAIT
    print("Voix du locuteur chargée.")
else:
    CANDIDATES = []
    print("Phrase d'essai :", PHRASE_TEST)
    for n in range(1, 5):
        torch.manual_seed(n)
        audio = model.generate(text=PHRASE_TEST, language="eto", num_step=32)[0]
        chemin = os.path.join(TRAVAIL, f"voix_candidate_{n}.wav")
        sf.write(chemin, audio, FREQ)
        CANDIDATES.append(chemin)
        print(f"Voix {n} :")
        display(Audio(audio, rate=FREQ))

### 6 bis. Valider la voix (option B seulement)
Indiquez le numéro de la voix choisie (1 à 4), puis exécutez.

In [ ]:
import shutil
VOIX_CHOISIE = 1

if not VOIX_LOCUTEUR:
    shutil.copy(CANDIDATES[VOIX_CHOISIE - 1], REFERENCE)
    TEXTE_REFERENCE = PHRASE_TEST
VOIX = model.create_voice_clone_prompt(ref_audio=REFERENCE, ref_text=TEXTE_REFERENCE or None)  # vide : transcription automatique (Whisper)
print("Voix fixée pour tous les clips.")

## 7. Générer tous les clips
Environ 1 à 3 secondes par clip sur un GPU T4. Les clips déjà faits sont gardés : en cas de coupure, relancez simplement cette cellule.
Chaque clip est enregistré en WAV puis converti en **OGG Opus** (fichier léger pour l'extension).

In [ ]:
import time
debut = time.time()
for i, t in enumerate(TEXTES, 1):
    ogg = os.path.join(OGG, t["id"] + ".ogg")
    if os.path.exists(ogg):
        continue
    torch.manual_seed(1234)
    audio = model.generate(text=t["text"], language="eto", voice_clone_prompt=VOIX, num_step=32)[0]
    wav = os.path.join(WAV, t["id"] + ".wav")
    sf.write(wav, audio, FREQ)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav, "-ac", "1", "-c:a", "libopus", "-b:a", "32k", ogg], check=True)
    print(f"[{i}/{len(TEXTES)}] {t['id']} — {len(audio) / FREQ:.1f} s")
print(f"Terminé en {time.time() - debut:.0f} s.")

### 7 bis. Écoute par le locuteur
Le locuteur écoute chaque clip. Dans le fichier **revue_locuteur.csv** (dans le dossier de travail), il note `oui` ou `non` dans la colonne « correct », et une remarque si besoin.
Seuls les clips marqués `oui` seront utilisés par l'extension.
Si la feuille existe déjà, ses réponses sont gardées pour les textes inchangés ; les textes nouveaux ou modifiés sont à écouter.

In [ ]:
import csv
revue = os.path.join(TRAVAIL, "revue_locuteur.csv")
avant = {}
if os.path.exists(revue):
    with open(revue, newline="", encoding="utf-8") as f:
        avant = {r["id"]: r for r in csv.DictReader(f)}
a_ecouter = []
with open(revue, "w", newline="", encoding="utf-8") as f:
    ecrit = csv.writer(f)
    ecrit.writerow(["id", "texte", "correct", "remarque"])
    for t in TEXTES:
        r = avant.get(t["id"])
        if r and r["texte"] == t["text"]:
            ecrit.writerow([t["id"], t["text"], r["correct"], r["remarque"]])
        else:
            ecrit.writerow([t["id"], t["text"], "", ""])
            a_ecouter.append(t)
print(len(a_ecouter), "clip(s) nouveaux ou modifiés à écouter :")
for t in a_ecouter:
    print(t["id"], "—", t["text"])
    display(Audio(os.path.join(WAV, t["id"] + ".wav")))

## 8. Télécharger le résultat
Crée **alpha_eton_omnivoice.zip** : les clips OGG, l'index `omnivoice-index.json` et la feuille de revue.
Sur votre PC, décompressez-le dans **data/audio/eton/omnivoice/** du projet ALPHA.

In [ ]:
import datetime
index = []
for t in TEXTES:
    ogg = os.path.join(OGG, t["id"] + ".ogg")
    if os.path.exists(ogg):
        index.append({
            "id": t["id"], "key": t["key"], "text": t["text"], "file": t["id"] + ".ogg",
            "source": "omnivoice", "voix": "locuteur (clonée)" if VOIX_LOCUTEUR else f"synthèse n°{VOIX_CHOISIE}",
        })
meta = {"modele": "k2-fsa/OmniVoice", "langue": "eto", "date": datetime.date.today().isoformat(), "clips": index}
with open(os.path.join(OGG, "omnivoice-index.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
shutil.copy(os.path.join(TRAVAIL, "revue_locuteur.csv"), os.path.join(OGG, "revue_locuteur.csv"))
archive = shutil.make_archive("/content/alpha_eton_omnivoice", "zip", OGG)
print(len(index), "clips dans", archive)
files.download(archive)